# V13 – Theorie: Rechnerarchitektur (Teil 1)

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- die **Von-Neumann-Architektur** mit ihren fünf Bausteinen benennen und erklären,
- den **Fetch-Decode-Execute-Zyklus** einer CPU in eigenen Worten beschreiben,
- die Rolle von **Registern** und der **Speicherhierarchie** (L1/L2/L3, RAM, SSD, HDD) einordnen,
- **Daten-**, **Adress-** und **Steuerbus** voneinander unterscheiden,
- den Unterschied zwischen einer klassischen **CPU**, einem **Mehrkern-Prozessor** und einer **GPU** skizzieren,
- begründen, warum diese Hardware-Ebene für **Simulation, Messtechnik und CNC** ingenieurs­relevant ist.

## ⏱️ Zeitbudget
Rund 35 Minuten.

## 🧭 TL;DR
Ein Rechner besteht aus **CPU**, **Speicher**, **Bus-System** und **I/O**. Die CPU führt Befehle im **Fetch-Decode-Execute-Zyklus** aus, greift dafür auf eine hierarchisch gestaffelte Speicherlandschaft zu und kommuniziert über Busse mit Peripherie. Wer diese Grundlagen versteht, kann Performance-Engpässe in Ingenieur-Anwendungen gezielt analysieren.

## 📦 Voraussetzungen
- Grundkenntnisse Binärzahlen (V01), Fließkommazahlen (V02)
- Python-Kenntnisse V05–V06 (siehe [00_python_recap.ipynb](00_python_recap.ipynb))

## Warum Rechnerarchitektur für Ingenieurinnen und Ingenieure?

Moderne Maschinenbau-Anwendungen sind ohne Rechner nicht denkbar. Eine **FEM-Simulation** zur Bauteilauslegung rechnet stundenlang auf spezialisierter Hardware, eine **CNC-Steuerung** muss Befehle in Mikrosekunden an Servomotoren weiterleiten, und eine **Messwerterfassung** an einem Prüfstand liest gleichzeitig hunderte Sensoren aus. All diese Anwendungen teilen sich dieselbe grundlegende Architektur – und wer sie versteht, erkennt schneller, **warum** ein Programm langsam läuft oder **welche** Hardware für eine Aufgabe geeignet ist.

In diesem Notebook steigst du deshalb von der obersten Ebene (Von-Neumann-Modell) schrittweise nach unten, bis hin zur Cache-Hierarchie und zum Bus-System. Wir bleiben dabei bewusst auf einem Niveau, das für die späteren Aufgaben und die Betriebssystem-Kapitel in V14 ausreicht.

## 1. Kurzer historischer Blick

Die erste Generation elektronischer Rechner (ENIAC, 1945) wurde noch durch manuelles Umstecken von Kabeln programmiert. Jede neue Aufgabe bedeutete tagelangen Umbau, und die physische Trennung zwischen „Programm" und „Daten" war eine Selbstverständlichkeit.

Im Jahr 1945 formulierte John **von Neumann** gemeinsam mit Kollegen am Institute for Advanced Study das Konzept eines **gespeicherten Programms**: Die Befehlsfolge sollte nicht länger über Schalter, sondern **als Zahlenfolge im selben Speicher** liegen wie die Daten. Diese Idee ist so einflussreich, dass nahezu jeder heutige Rechner – vom Smartphone bis zum Supercomputer – als Variante dieser Architektur gelten kann.

## 2. Die Von-Neumann-Architektur im Überblick

Die **Von-Neumann-Architektur** beschreibt ein Rechnerkonzept, in dem **Programm und Daten gemeinsam** in einem einheitlichen Speicher liegen. Die zentrale Recheneinheit (CPU) holt sich sowohl den nächsten Befehl als auch die zu verarbeitenden Werte aus diesem gemeinsamen Speicher. Verbunden sind alle Bausteine über ein zentrales **Bussystem**, das sowohl Adressen als auch Daten und Steuersignale überträgt.

Das nachfolgende Diagramm zeigt die fünf Hauptkomponenten – **Eingabewerk**, **Ausgabewerk**, **Speicher**, **Rechenwerk** und **Steuerwerk** – sowie ihre Verbindung über den **Systembus**.

📊 Diagramm-Quelle: [diagramme/01_von_neumann.mmd](diagramme/01_von_neumann.mmd)

In [ ]:
from IPython.display import Markdown, display

with open("diagramme/01_von_neumann.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))

## 2.1 Die fünf Bausteine im Detail

Damit aus dem abstrakten Diagramm ein konkretes mentales Modell entsteht, schauen wir uns jeden Baustein einzeln an. Jeder hat eine **klar umrissene Aufgabe** und einen typischen Industriebezug, den du dir als Anker merken kannst.

### Eingabewerk

Das **Eingabewerk** überträgt Daten aus der Außenwelt in den Speicher des Rechners. Im Büro-Alltag sind das Tastatur und Maus, im Ingenieur-Kontext dagegen **Messkarten** (z. B. National Instruments oder Beckhoff), die Spannungen von Dehnungsmessstreifen, Thermoelementen oder Beschleunigungssensoren digitalisieren. Alle diese Geräte haben eine Gemeinsamkeit: Sie liefern einen **Strom von Zahlen**, der in den Hauptspeicher geschrieben wird, bevor die CPU damit rechnen kann.

### Ausgabewerk

Analog zum Eingabewerk leitet das **Ausgabewerk** Daten aus dem Rechner nach außen. Das klassische Beispiel ist der Bildschirm, im Maschinenbau sind es jedoch häufig **Aktoren**: Servomotoren einer CNC-Fräse, Magnetventile einer Hydraulikanlage oder Stellglieder eines Regelkreises. Die CPU schreibt dafür Zahlen in eine Speicheradresse, die physikalisch mit dem Ausgabegerät verknüpft ist („memory-mapped I/O"), und das Gerät übersetzt diese Zahl in eine Spannung oder einen digitalen Impuls.

### Speicher

Der **Hauptspeicher** (RAM) hält während des Programmlaufs sowohl die Befehle als auch die bearbeiteten Daten. Er ist als lineares Array von **Zellen** organisiert, die jeweils eine Adresse haben und typischerweise **1 Byte** (8 Bit) groß sind. Die CPU adressiert eine Zelle über ihren Index, liest den Inhalt oder schreibt ihn zurück.

> [!NOTE]
> Wenn du im Python-Debugger eine Variable siehst, liegt ihr Wert tatsächlich an einer ganz konkreten Adresse im RAM. Die Funktion `id(x)` aus V03 liefert dir in CPython sogar einen Zeiger darauf.

### Rechenwerk (ALU)

Das **Rechenwerk**, oft als **ALU** (Arithmetic Logic Unit) bezeichnet, führt die eigentlichen Berechnungen aus: Addition, Subtraktion, Multiplikation, logische `AND`/`OR`/`NOT`-Verknüpfungen und Vergleiche. Für Fließkomma-Operationen existiert üblicherweise ein zusätzliches **Gleitkommawerk** (FPU), das beispielsweise die in V02 besprochenen IEEE-754-Zahlen in wenigen Nanosekunden multipliziert. Moderne CPUs kombinieren mehrere ALUs und spezielle **Vektoreinheiten** (SIMD, z. B. AVX-512), die dieselbe Operation gleichzeitig auf 8 oder 16 Werte anwenden – essenziell für FEM-Löser und Signalverarbeitung.

### Steuerwerk

Das **Steuerwerk** ist der Dirigent der CPU: Es liest Befehle aus dem Speicher, entschlüsselt sie und aktiviert die passenden Leitungen, damit die ALU rechnet, Register gelesen oder geschrieben werden und gegebenenfalls der nächste Befehl aus einer anderen Speicherstelle geholt wird. In modernen Prozessoren ist das Steuerwerk erstaunlich komplex – es beherrscht **Pipelining**, **Sprungvorhersage** und **Out-of-Order-Execution**, um die ALUs möglichst selten leer laufen zu lassen.

Für die Vorlesung genügt die Vorstellung: Das Steuerwerk weiß in jedem Taktzyklus genau, **welcher Befehl gerade dran ist**, und schickt den Rest der CPU entsprechend an die Arbeit.

> [!NOTE]
> **💡 Weiterführend: Harvard- vs. Von-Neumann-Architektur.** In der Von-Neumann-Variante teilen sich Programm und Daten denselben Speicher und denselben Bus. Die **Harvard-Architektur** trennt beides physisch, was in kleinen Mikrocontrollern (z. B. AVR in Arduino-Boards) und digitalen Signalprozessoren häufig vorkommt. Der Vorteil: Befehle und Daten können parallel geholt werden. Der Nachteil: Unflexiblere Speicherverwaltung. Moderne Desktop-CPUs nutzen einen Hybrid – getrennte L1-Caches für Befehle und Daten, aber gemeinsamer Hauptspeicher.

## 3. Die CPU im Detail

Die **Central Processing Unit** (CPU) ist das Herzstück jedes Rechners und vereint Rechenwerk und Steuerwerk auf einem einzigen Chip. Moderne CPUs enthalten außerdem mehrere Cache-Ebenen, Speicher-Controller und Schnittstellen zu Peripherie-Bussen direkt auf dem Silizium-Die. Was du als Endnutzerin oder Endnutzer kaufst – einen Intel Core i7 oder einen AMD Ryzen 7 – ist also bereits ein kleines System-on-a-Chip.

In diesem Abschnitt konzentrieren wir uns auf die **logische Funktionsweise**: Wie verarbeitet die CPU einen einzelnen Befehl, und warum ist dieser Ablauf für das Verständnis von Performance so zentral?

### Der Fetch-Decode-Execute-Zyklus

Jeder Befehl durchläuft in der CPU denselben drei- bis vierstufigen Ablauf, den man **Fetch-Decode-Execute-Zyklus** (oft erweitert um einen Write-Back-Schritt) nennt. Die folgende Abbildung zeigt die vier Phasen und den Schleifencharakter: Sobald ein Befehl fertig ist, wird der **Program Counter** (PC) erhöht und der nächste Befehl geladen.

📊 Diagramm-Quelle: [diagramme/04_cpu_zyklus.mmd](diagramme/04_cpu_zyklus.mmd)

In [ ]:
from IPython.display import Markdown, display

with open("diagramme/04_cpu_zyklus.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))

Im Detail laufen die vier Phasen wie folgt ab:

1. **Fetch** – Das Steuerwerk liest die im Program Counter hinterlegte Adresse, holt den dort liegenden Befehl aus dem Speicher und legt ihn ins **Befehlsregister** (Instruction Register, IR).
2. **Decode** – Der Befehl wird analysiert: Welche Operation (`ADD`, `LOAD`, `JUMP`) ist gemeint? Welche Register oder Speicheradressen sind Operanden?
3. **Execute** – Die ALU führt die eigentliche Rechenoperation aus oder das Steuerwerk initiiert einen Speicherzugriff.
4. **Write-Back** – Das Ergebnis wird zurück in ein Register oder in den Hauptspeicher geschrieben, und der Program Counter wird auf den nächsten Befehl erhöht.

Diese vier Schritte wiederholen sich milliardenfach pro Sekunde – eine CPU mit 3 GHz arbeitet rund 3 Milliarden solcher Zyklen pro Sekunde ab, wobei durch **Pipelining** mehrere Befehle gleichzeitig in unterschiedlichen Phasen bearbeitet werden.

### Takt und Taktfrequenz

Die **Taktfrequenz** gibt an, wie viele elementare Zyklen pro Sekunde ablaufen, und wird in Hertz gemessen. Eine Angabe wie `3,6 GHz` bedeutet `3,6 × 10⁹` Zyklen pro Sekunde. Nicht jeder Befehl ist in einem Zyklus fertig – eine Division benötigt typisch 10–40 Zyklen, ein einfacher Register-Vergleich nur einen.

Die folgende kleine Simulation zeigt dir plastisch, wie viele Operationen moderne CPUs in Sekundenbruchteilen erledigen: Wir messen, wie lange Python für eine Million einfacher Additionen braucht.

In [5]:
import time

n = 1_000_000
start = time.perf_counter()
summe = 0
for i in range(n):
    summe += i
dauer_ms = (time.perf_counter() - start) * 1000

print(f"{n} Additionen in {dauer_ms:.1f} ms")
print(f"Das sind rund {n / dauer_ms:.0f} Operationen pro Millisekunde.")

1000000 Additionen in 121.4 ms
Das sind rund 8236 Operationen pro Millisekunde.


## 4. Register – das schnellste Gedächtnis der CPU

**Register** sind winzige Speicherplätze **direkt in der CPU**, typischerweise 64 Bit breit. Sie sind so schnell, dass die ALU in einem einzigen Taktzyklus darauf zugreifen kann. Eine typische x86-64-CPU hat rund 16 allgemein nutzbare Register für Ganzzahlen und weitere 16–32 für Fließkomma- und Vektoroperationen.

### Wichtige Register-Typen

Auch wenn du in Python selten direkt mit Registern zu tun hast, ist es hilfreich, ihre Rollen zu kennen:

- **Program Counter (PC)** – enthält die Adresse des nächsten auszuführenden Befehls.
- **Instruction Register (IR)** – speichert den aktuell dekodierten Befehl.
- **Akkumulator / Universalregister** – nehmen Zwischenergebnisse auf (heute meistens `RAX`, `RBX`, … auf x86-64).
- **Stackpointer (SP)** – zeigt auf das obere Ende des Call-Stacks und steuert Funktionsaufrufe.
- **Status- oder Flag-Register** – halten einzelne Bits für Ergebnisinformationen wie „Resultat ist Null", „Überlauf aufgetreten", „Vorzeichen negativ".

### Warum Register so wichtig sind

Ein Registerzugriff dauert typisch **unter 1 Nanosekunde**, ein RAM-Zugriff dagegen **rund 100 Nanosekunden**. Der Unterschied liegt also bei Faktor 100. Wenn die ALU in jedem Taktzyklus einen neuen Operanden aus dem RAM holen müsste, liefe sie 99 % der Zeit leer. Genau deshalb versucht der **Compiler** (und bei interpretierten Sprachen wie Python der Python-Interpreter samt JIT oder C-Backend), möglichst viele Operanden in Registern zu halten. In NumPy oder SciPy-Code, der die meisten Ingenieur-Probleme trifft, geschieht das bereits automatisch.